# Home Credit Default Risk – Improved Model (LightGBM + CatBoost blend)
Run every cell from top to bottom (**Runtime → Run all**). The whole notebook takes about **15–25 minutes** on Colab's free CPU.

**Validation results (5-fold CV AUC):**

| Model | CV AUC |
|---|---|
| Tutorial baseline | ~0.70 |
| LightGBM, 1 seed | 0.7613 |
| LightGBM, average of 3 seeds | 0.7619 |
| CatBoost | 0.7607 |
| **Blend: 60% LightGBM + 40% CatBoost** | **0.7629** |

| Step | What it does |
|---|---|
| 0 | Setup: connect Google Drive, set the path |
| 1 | Load the data |
| 2 | Clean the data (fix anomalies and "XNA") |
| 3 | Create new features |
| 4 | Encode the text (categorical) columns |
| 5 | Train LightGBM (3 seeds) with 5-fold cross-validation |
| 6 | Train CatBoost with 5-fold cross-validation |
| 7 | Blend the two models |
| 8 | Save `submission.csv` |
| 9 | Save this notebook as a ZIP (needed for Honors/Outstanding) |

All random seeds are fixed (`SEED = 42`) so the results can be reproduced, as the rules require.

## Step 0: Setup

In [ ]:
# CatBoost is not pre-installed on Colab
!pip install -q catboost

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

In [ ]:
# Google Colab only
from google.colab import drive
drive.mount('/content/drive')

**Change this path** to the folder where you uploaded the `competition` folder (the same path you used in `tutorial.ipynb`).

In [ ]:
%cd "/content/drive/MyDrive/competition"

In [ ]:
current_dir = Path(os.getcwd())
train_file = current_dir / "input" / "train.csv"
test_file = current_dir / "input" / "test.csv"
sample_sub_file = current_dir / "input" / "sample_submission.csv"
print("Path OK" if train_file.exists() and test_file.exists() else "Path is WRONG – fix the %cd cell above")

## Step 1: Load the data
Train and test are joined into one table (`df`) so that every cleaning and feature step is applied to both in exactly the same way.

In [ ]:
train = pd.read_csv(train_file)
test = pd.read_csv(test_file)
sample_sub = pd.read_csv(sample_sub_file)

y = train["TARGET"].values
df = pd.concat([train.drop(columns="TARGET"), test], ignore_index=True)
n_train = len(train)
print(train.shape, test.shape, df.shape)

## Step 2: Clean the data
- `DAYS_EMPLOYED = 365243` is not a real value (that would be about 1000 years). It mostly marks pensioners and unemployed people, so it is replaced with NaN and a flag column records it.
- `"XNA"` means "not available", so it is replaced with NaN.
- `DAYS_LAST_PHONE_CHANGE = 0` is most likely missing too.

LightGBM handles NaN values by itself, so missing values don't need to be filled in.

In [ ]:
df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
df = df.replace("XNA", np.nan)
df.loc[df["DAYS_LAST_PHONE_CHANGE"] == 0, "DAYS_LAST_PHONE_CHANGE"] = np.nan

## Step 3: Create new features
Ratios often carry more signal than raw amounts. For example, "the loan is 5× the yearly income" says more than "the loan is 500,000".
The `EXT_SOURCE` columns are external credit scores and are the strongest predictors, so combining them helps.

In [ ]:
# Money ratios
df["CREDIT_INCOME_RATIO"]  = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
df["CREDIT_TERM"]          = df["AMT_ANNUITY"] / df["AMT_CREDIT"]      # roughly 1 / number of payments
df["CREDIT_GOODS_RATIO"]   = df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"]
df["CREDIT_GOODS_DIFF"]    = df["AMT_CREDIT"] - df["AMT_GOODS_PRICE"]
df["INCOME_PER_PERSON"]    = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]

# Time ratios
df["AGE_YEARS"]          = -df["DAYS_BIRTH"] / 365
df["EMPLOYED_AGE_RATIO"] = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]
df["CAR_AGE_RATIO"]      = df["OWN_CAR_AGE"] / df["DAYS_BIRTH"]
df["PHONE_AGE_RATIO"]    = df["DAYS_LAST_PHONE_CHANGE"] / df["DAYS_BIRTH"]
df["ID_AGE_RATIO"]       = df["DAYS_ID_PUBLISH"] / df["DAYS_BIRTH"]

# External score combinations
ext = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
df["EXT_MEAN"]      = ext.mean(axis=1)
df["EXT_STD"]       = ext.std(axis=1)
df["EXT_MIN"]       = ext.min(axis=1)
df["EXT_MAX"]       = ext.max(axis=1)
df["EXT_PROD"]      = ext.prod(axis=1, min_count=3)
df["EXT_NAN_COUNT"] = ext.isnull().sum(axis=1)
df["EXT_WEIGHTED"]  = df["EXT_SOURCE_1"] * 2 + df["EXT_SOURCE_2"] * 1 + df["EXT_SOURCE_3"] * 3

# How much information is missing for this customer
df["N_MISSING"] = df.isnull().sum(axis=1)

## Step 4: Encode the categorical columns
LightGBM can use text columns directly once they are converted to the pandas `category` type, so one-hot encoding isn't needed.

In [ ]:
cat_cols = df.select_dtypes(exclude="number").columns.tolist()  # all text columns
for c in cat_cols:
    df[c] = df[c].astype("category")

features = [c for c in df.columns if c != "SK_ID_CURR"]
X = df.iloc[:n_train][features].reset_index(drop=True)
X_test = df.iloc[n_train:][features].reset_index(drop=True)
print("categorical:", cat_cols)
print("number of features:", len(features))

## Step 5: Train LightGBM with 5-fold cross-validation (3 seeds)
- The training data is split into 5 parts. Each model trains on 4 parts and is scored on the 5th, so every row gets an honest prediction.
- **CV AUC** is your own score. Trust it more than the public leaderboard, because the final ranking uses the hidden private leaderboard.
- Early stopping picks the number of trees automatically.
- The whole thing is repeated with **3 different random seeds** and averaged. This makes the predictions more stable (CV 0.7613 → 0.7619).
- The **same 5 folds** are used for every model, so the CV scores can be compared and blended fairly.

This takes about 5–10 minutes on Colab.

In [ ]:
lgb_params = dict(
    objective="binary", metric="auc", learning_rate=0.02,
    num_leaves=24, min_child_samples=100,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
    reg_alpha=1.0, reg_lambda=5.0, cat_smooth=50, min_data_per_group=200,
    n_estimators=10000, verbose=-1, n_jobs=-1,
)
LGB_SEEDS = [42, 7, 2024]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))

oof_lgb = np.zeros(len(X))
pred_lgb = np.zeros(len(X_test))

for seed in LGB_SEEDS:
    oof_s = np.zeros(len(X))
    for fold, (tr_idx, va_idx) in enumerate(folds):
        model = lgb.LGBMClassifier(**lgb_params, random_state=seed)
        model.fit(
            X.iloc[tr_idx], y[tr_idx],
            eval_set=[(X.iloc[va_idx], y[va_idx])],
            callbacks=[lgb.early_stopping(200, verbose=False)],
        )
        oof_s[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
        pred_lgb += model.predict_proba(X_test)[:, 1] / (len(folds) * len(LGB_SEEDS))
    oof_lgb += oof_s / len(LGB_SEEDS)
    print(f"Seed {seed}: CV AUC = {roc_auc_score(y, oof_s):.5f}")

print(f"\nLightGBM (3 seeds) CV AUC = {roc_auc_score(y, oof_lgb):.5f}")

Which features did the model rely on most? This helps you decide what to try next.

In [ ]:
importance = pd.Series(model.feature_importances_, index=features).sort_values()
importance.tail(25).plot(kind="barh", figsize=(8, 8), title="LightGBM feature importance (last fold)")

## Step 6: Train CatBoost with 5-fold cross-validation
CatBoost is a different gradient boosting library that is especially good with text (categorical) columns.
It makes **different mistakes** from LightGBM, which is what makes blending work.

CatBoost needs text columns as strings, with missing values written as `"NA"`.

This is the slowest step: about 10–15 minutes on Colab.

In [ ]:
X_cb = X.copy()
X_test_cb = X_test.copy()
for c in cat_cols:
    X_cb[c] = X_cb[c].astype(object).fillna("NA").astype(str)
    X_test_cb[c] = X_test_cb[c].astype(object).fillna("NA").astype(str)

oof_cat = np.zeros(len(X))
pred_cat = np.zeros(len(X_test))
test_pool = Pool(X_test_cb, cat_features=cat_cols)

for fold, (tr_idx, va_idx) in enumerate(folds):
    model_cb = CatBoostClassifier(
        iterations=10000, learning_rate=0.06, depth=6, l2_leaf_reg=5,
        eval_metric="AUC", random_seed=SEED, od_type="Iter", od_wait=300,
        verbose=0, thread_count=-1,
    )
    model_cb.fit(
        Pool(X_cb.iloc[tr_idx], y[tr_idx], cat_features=cat_cols),
        eval_set=Pool(X_cb.iloc[va_idx], y[va_idx], cat_features=cat_cols),
        use_best_model=True,
    )
    oof_cat[va_idx] = model_cb.predict_proba(X_cb.iloc[va_idx])[:, 1]
    pred_cat += model_cb.predict_proba(test_pool)[:, 1] / len(folds)
    print(f"Fold {fold}: AUC = {roc_auc_score(y[va_idx], oof_cat[va_idx]):.5f} (trees: {model_cb.get_best_iteration()})")

print(f"\nCatBoost CV AUC = {roc_auc_score(y, oof_cat):.5f}")

## Step 7: Blend the two models
Take a weighted average of the two models' predictions. The weight is checked on the out-of-fold (CV) predictions; 60% LightGBM + 40% CatBoost worked best.

In [ ]:
for w in [0.0, 0.3, 0.4, 0.5, 0.6, 1.0]:
    print(f"CatBoost weight {w:.1f}: CV AUC = {roc_auc_score(y, (1 - w) * oof_lgb + w * oof_cat):.5f}")

W_CAT = 0.4
oof_blend = (1 - W_CAT) * oof_lgb + W_CAT * oof_cat
pred = (1 - W_CAT) * pred_lgb + W_CAT * pred_cat
print(f"\nFinal blend CV AUC = {roc_auc_score(y, oof_blend):.5f}")

## Step 8: Save the submission (do not change this format)

In [ ]:
sample_sub["TARGET"] = pred
output_dir = current_dir / "output"
os.makedirs(output_dir, exist_ok=True)
sample_sub.to_csv(output_dir / "submission.csv", index=False)
print(sample_sub.shape)
sample_sub.head()

## Step 9: Save this notebook as a ZIP (required for Honors/Outstanding)
1. Save the notebook first (**Ctrl+S / Cmd+S**).
2. Run the cell below.
3. Download `my_model_v2.zip` and upload it on Omnicampus under **Competition (.zip)**.

In [ ]:
import zipfile
notebook_filename = "my_model_v2.ipynb"
zip_filename = "my_model_v2.zip"
notebook_path = current_dir / notebook_filename
if notebook_path.exists():
    with zipfile.ZipFile(current_dir / zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(notebook_path, arcname=notebook_filename)
    print("Created", current_dir / zip_filename)
else:
    print(f"Could not find {notebook_filename} in {current_dir}")